# =============================================================================
# HYPERPARAMETER OPTIMIZATION WITH OPTUNA
# =============================================================================
# 
# Automated hyperparameter search for best Custom CNN architecture
# Using Optuna for intelligent Bayesian optimization
# 
# Target: Optimize RGB Custom CNN (current best: 80.0%)
# 
# Author: Pavlo Borysov
# Date: 13/10/2025
# =============================================================================


In [10]:
# =============================================================================
# HPO CONFIGURATION
# =============================================================================

# Study settings
N_TRIALS = 40  # More trials for refined search
STUDY_NAME = "fer_baseline_refinement"
STORAGE_FILE = "../../runs/hpo/optuna_baseline_refinement.db"  # New storage

# BASELINE PARAMETERS (80.0% test accuracy)
BASELINE = {
    "dropout": 0.3,
    "l2_reg": 0.001,
    "lr": 0.0001,
    "batch_size": 64,
    "augmentation": "medium"
}

# Search space - AROUND BASELINE (±50% variation)
SEARCH_SPACE = {
    # Architecture - Fixed to proven best
    "architecture": ["complex"],
    "channels_multiplier": [0.8, 1.0, 1.2],  # ±20% around baseline
    "convs_per_block": [2],  # Fixed
    
    # Regularization - AROUND BASELINE
    "dropout_min": 0.2,      # 0.3 - 33%
    "dropout_max": 0.45,     # 0.3 + 50%
    "l2_min": 0.0005,        # 0.001 - 50%
    "l2_max": 0.002,         # 0.001 + 100%
    
    # Training - AROUND BASELINE
    "lr_min": 0.00005,       # 0.0001 - 50%
    "lr_max": 0.0003,        # 0.0001 + 200%
    "batch_sizes": [64, 128, 256],  # Include baseline + larger
    
    # Augmentation - ALL OPTIONS
    "augmentation_options": ["light", "medium", "strong"]
}

# Fixed settings
MODEL_TYPE = "rgb"
IMG_SIZE = (48, 48)
EPOCHS = 20  # Faster convergence check
PATIENCE = 5  # More aggressive early stopping

print("=" * 80)
print("OPTUNA HPO - BASELINE REFINEMENT")
print("=" * 80)
print(f"Strategy: Search AROUND baseline (80.0% accuracy)")
print(f"\nBaseline parameters:")
for key, val in BASELINE.items():
    print(f"  {key:15s}: {val}")
print(f"\nStudy configuration:")
print(f"  Name: {STUDY_NAME}")
print(f"  Trials: {N_TRIALS}")
print(f"  Epochs/trial: {EPOCHS}")
print(f"  Expected time: ~{N_TRIALS * 2} minutes ({N_TRIALS * 2 / 60:.1f} hours)")
print(f"\nSearch Space (around baseline):")
print(f"  Channels mult: {SEARCH_SPACE['channels_multiplier']} (baseline=1.0)")
print(f"  Dropout: {SEARCH_SPACE['dropout_min']:.2f}-{SEARCH_SPACE['dropout_max']:.2f} (baseline=0.3)")
print(f"  L2 reg: {SEARCH_SPACE['l2_min']:.4f}-{SEARCH_SPACE['l2_max']:.4f} (baseline=0.001)")
print(f"  LR: {SEARCH_SPACE['lr_min']:.5f}-{SEARCH_SPACE['lr_max']:.5f} (baseline=0.0001)")
print(f"  Batch: {SEARCH_SPACE['batch_sizes']} (baseline=64)")
print(f"  Augmentation: {SEARCH_SPACE['augmentation_options']} (baseline=medium)")
print("=" * 80)


OPTUNA HPO - BASELINE REFINEMENT
Strategy: Search AROUND baseline (80.0% accuracy)

Baseline parameters:
  dropout        : 0.3
  l2_reg         : 0.001
  lr             : 0.0001
  batch_size     : 64
  augmentation   : medium

Study configuration:
  Name: fer_baseline_refinement
  Trials: 40
  Epochs/trial: 20
  Expected time: ~80 minutes (1.3 hours)

Search Space (around baseline):
  Channels mult: [0.8, 1.0, 1.2] (baseline=1.0)
  Dropout: 0.20-0.45 (baseline=0.3)
  L2 reg: 0.0005-0.0020 (baseline=0.001)
  LR: 0.00005-0.00030 (baseline=0.0001)
  Batch: [64, 128, 256] (baseline=64)
  Augmentation: ['light', 'medium', 'strong'] (baseline=medium)


In [11]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================

import warnings
import os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import time
import json
from pathlib import Path

from sklearn.metrics import f1_score

import tensorflow as tf
import keras
from keras import layers
from keras.utils import image_dataset_from_directory

# Optuna for HPO
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

# Fixed random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Configure GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")

print(f"TensorFlow: {tf.__version__}")
print(f"Optuna: {optuna.__version__}")
print(f"GPU: {gpus}")


ModuleNotFoundError: No module named 'optuna'

In [8]:
# =============================================================================
# PATHS AND CONSTANTS
# =============================================================================

# Data paths
TRAIN_DATA_DIR = Path("../../data/train")
VALIDATION_DATA_DIR = Path("../../data/validation")
TEST_DATA_DIR = Path("../../data/test")

# Output paths
HPO_DIR = Path("../../runs/hpo")
HPO_DIR.mkdir(parents=True, exist_ok=True)

# Fixed class order and weights
CLASS_ORDER = ['happy', 'neutral', 'sad', 'surprise']
CLASS_WEIGHTS = {
    0: 0.950,  # happy
    1: 0.950,  # neutral  
    2: 0.949,  # sad
    3: 1.190   # surprise
}

NUM_CLASSES = 4
METRIC_NAME = "sparse_categorical_accuracy"
VAL_METRIC_NAME = f"val_{METRIC_NAME}"

print(f"HPO directory: {HPO_DIR}")
print(f"Class order: {CLASS_ORDER}")


HPO directory: ..\..\runs\hpo
Class order: ['happy', 'neutral', 'sad', 'surprise']


In [4]:
# =============================================================================
# DATA LOADERS WITH CONFIGURABLE AUGMENTATION
# =============================================================================

def make_generators(augmentation_strength="medium", batch_size=64):
    """Create data loaders with configurable augmentation."""
    
    def preprocess_image(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    
    def augment_light(image, label):
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.1)
        image = tf.clip_by_value(image, 0.0, 1.0)
        return image, label
    
    def augment_medium(image, label):
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.2)
        image = tf.image.random_contrast(image, 0.8, 1.2)
        image = tf.clip_by_value(image, 0.0, 1.0)
        return image, label
    
    def augment_strong(image, label):
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.3)
        image = tf.image.random_contrast(image, 0.7, 1.3)
        # Add slight rotation
        image = tf.clip_by_value(image, 0.0, 1.0)
        return image, label
    
    # Select augmentation
    aug_map = {"light": augment_light, "medium": augment_medium, "strong": augment_strong}
    augment_fn = aug_map[augmentation_strength]
    
    # Train data
    train_ds = image_dataset_from_directory(
        TRAIN_DATA_DIR,
        class_names=CLASS_ORDER,
        image_size=IMG_SIZE,
        batch_size=batch_size,
        color_mode="rgb",
        shuffle=True
    )
    train_ds = train_ds.map(preprocess_image).map(augment_fn)
    
    # Validation data
    val_ds = image_dataset_from_directory(
        VALIDATION_DATA_DIR,
        class_names=CLASS_ORDER,
        image_size=IMG_SIZE,
        batch_size=batch_size,
        color_mode="rgb",
        shuffle=False
    )
    val_ds = val_ds.map(preprocess_image)
    
    return train_ds, val_ds

print("Data loader functions ready!")


Data loader functions ready!


In [5]:
# =============================================================================
# MODEL BUILDER
# =============================================================================

def build_model(channels, convs_per_block, dropout, l2_reg, lr):
    """Build CNN model with given hyperparameters."""
    
    input_shape = IMG_SIZE + (3,)
    
    inputs = keras.Input(shape=input_shape, name="input")
    x = inputs
    
    # Convolutional blocks
    for i, filters in enumerate(channels):
        for j in range(convs_per_block):
            x = layers.Conv2D(
                filters, 3, padding="same",
                kernel_regularizer=keras.regularizers.l2(l2_reg),
                name=f"conv_{i+1}_{j+1}"
            )(x)
            x = layers.BatchNormalization(name=f"bn_{i+1}_{j+1}")(x)
            x = layers.LeakyReLU(name=f"leaky_{i+1}_{j+1}")(x)
        
        # Max pooling (except last block)
        if i < len(channels) - 1:
            x = layers.MaxPooling2D(2, name=f"pool_{i+1}")(x)
    
    # Head
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(dropout, name="dropout_head")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    
    model = keras.Model(inputs, outputs, name="emotion_cnn")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=[METRIC_NAME]
    )
    
    return model

print("Model builder ready!")


Model builder ready!


In [6]:
# =============================================================================
# OPTUNA OBJECTIVE FUNCTION
# =============================================================================

def objective(trial):
    """Optuna objective function to minimize."""
    
    # Sample hyperparameters - FROM SEARCH_SPACE CONFIG
    architecture_type = trial.suggest_categorical("architecture", SEARCH_SPACE["architecture"])
    channels_mult = trial.suggest_categorical("channels_multiplier", SEARCH_SPACE["channels_multiplier"])
    convs_per_block = SEARCH_SPACE["convs_per_block"][0]  # Fixed value from config
    
    # Define channels based on architecture and multiplier
    base_channels = (64, 128, 512, 512, 128)  # Complex architecture
    channels = tuple(int(c * channels_mult) for c in base_channels)
    
    # Regularization - FROM CONFIG
    dropout = trial.suggest_float("dropout", SEARCH_SPACE["dropout_min"], SEARCH_SPACE["dropout_max"])
    l2_reg = trial.suggest_float("l2_reg", SEARCH_SPACE["l2_min"], SEARCH_SPACE["l2_max"], log=True)
    
    # Training - FROM CONFIG
    lr = trial.suggest_float("learning_rate", SEARCH_SPACE["lr_min"], SEARCH_SPACE["lr_max"], log=True)
    batch_size = trial.suggest_categorical("batch_size", SEARCH_SPACE["batch_sizes"])
    
    # Augmentation - FROM CONFIG
    aug_strength = trial.suggest_categorical("augmentation", SEARCH_SPACE["augmentation_options"])
    
    # Log trial parameters
    print(f"\n{'='*70}")
    print(f"Trial {trial.number} Parameters:")
    print(f"  Architecture: {architecture_type}, channels: {channels}")
    print(f"  Dropout: {dropout:.3f}, L2: {l2_reg:.6f}")
    print(f"  LR: {lr:.6f}, Batch: {batch_size}")
    print(f"  Augmentation: {aug_strength}")
    print(f"{'='*70}\n")
    
    # Clear previous model from memory
    keras.backend.clear_session()
    
    # Create data loaders
    train_ds, val_ds = make_generators(augmentation_strength=aug_strength, batch_size=batch_size)
    
    # Build model
    model = build_model(
        channels=channels,
        convs_per_block=convs_per_block,
        dropout=dropout,
        l2_reg=l2_reg,
        lr=lr
    )
    
    # Callbacks with Optuna pruning
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor=VAL_METRIC_NAME,
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1  # Show when stopping
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=VAL_METRIC_NAME,
            factor=0.5,
            patience=PATIENCE//2,
            min_lr=1e-7,
            verbose=1  # Show LR changes
        ),
        optuna.integration.TFKerasPruningCallback(trial, VAL_METRIC_NAME)
    ]
    
    # Train model with verbose output
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=CLASS_WEIGHTS,
        verbose=1  # Show training progress!
    )
    
    # Get best validation accuracy
    best_val_acc = max(history.history[VAL_METRIC_NAME])
    
    print(f"Trial {trial.number} Result: {best_val_acc:.4f}")
    
    # Return negative because Optuna minimizes by default
    # (we want to maximize accuracy)
    return -best_val_acc

print("Objective function ready!")


Objective function ready!


In [7]:
# =============================================================================
# RUN OPTUNA OPTIMIZATION
# =============================================================================

# Create study with pruning
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STORAGE_FILE}",
    load_if_exists=True,
    direction="minimize",  # Minimize negative accuracy
    sampler=TPESampler(seed=RANDOM_SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
)

print("=" * 80)
print(f"STARTING OPTUNA OPTIMIZATION")
print("=" * 80)
print(f"Study: {STUDY_NAME}")
print(f"Trials: {N_TRIALS}")
print(f"Storage: {STORAGE_FILE}")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

# Run optimization
start_time = time.time()

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True  # Clean memory after each trial
)

total_time = time.time() - start_time

print("\n" + "=" * 80)
print("OPTIMIZATION COMPLETED!")
print("=" * 80)
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"Completed trials: {len(study.trials)}")
print(f"Pruned trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"Best trial: {study.best_trial.number}")
print(f"Best value (negative accuracy): {study.best_value:.4f}")
print(f"Best accuracy: {-study.best_value:.4f}")
print("=" * 80)


[I 2025-10-13 07:31:53,818] A new study created in RDB with name: fer_baseline_refinement


STARTING OPTUNA OPTIMIZATION
Study: fer_baseline_refinement
Trials: 40
Storage: ../../runs/hpo/optuna_baseline_refinement.db
Start time: 2025-10-13 07:31:53


  0%|          | 0/40 [00:00<?, ?it/s]


Trial 0 Parameters:
  Architecture: complex, channels: (64, 128, 512, 512, 128)
  Dropout: 0.350, L2: 0.000621
  LR: 0.000066, Batch: 128
  Augmentation: strong

Found 15109 files belonging to 4 classes.
Found 4977 files belonging to 4 classes.
Epoch 1/20


I0000 00:00:1760340744.294563   10658 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


119/119 [==============================] - 19s 90ms/step - loss: 2.5869 - sparse_categorical_accuracy: 0.4983 - val_loss: 2.8805 - val_sparse_categorical_accuracy: 0.2289 - lr: 6.6124e-05
Epoch 2/20
119/119 [==============================] - 10s 81ms/step - loss: 2.3175 - sparse_categorical_accuracy: 0.6247 - val_loss: 3.0090 - val_sparse_categorical_accuracy: 0.2305 - lr: 6.6124e-05
Epoch 3/20
119/119 [==============================] - 9s 72ms/step - loss: 2.1922 - sparse_categorical_accuracy: 0.6715 - val_loss: 2.7168 - val_sparse_categorical_accuracy: 0.3580 - lr: 6.6124e-05
Epoch 4/20
119/119 [==============================] - 9s 69ms/step - loss: 2.1042 - sparse_categorical_accuracy: 0.7106 - val_loss: 2.3927 - val_sparse_categorical_accuracy: 0.5680 - lr: 6.6124e-05
Epoch 5/20
119/119 [==============================] - 9s 75ms/step - loss: 2.0122 - sparse_categorical_accuracy: 0.7441 - val_loss: 2.5004 - val_sparse_categorical_accuracy: 0.5258 - lr: 6.6124e-05
Epoch 6/20
119/119 

In [13]:
# =============================================================================
# BEST PARAMETERS AND ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("BEST HYPERPARAMETERS")
print("=" * 80)

best_params = study.best_trial.params
for param, value in best_params.items():
    print(f"{param:20s}: {value}")

print("=" * 80)

# Save best parameters
best_params_file = HPO_DIR / "best_parameters.json"
with open(best_params_file, "w") as f:
    json.dump({
        "best_trial_number": study.best_trial.number,
        "best_accuracy": -study.best_value,
        "best_params": best_params,
        "optimization_time_minutes": total_time / 60
    }, f, indent=2)

print(f"\nBest parameters saved to: {best_params_file}")



BEST HYPERPARAMETERS


NameError: name 'study' is not defined

In [12]:
# =============================================================================
# MATPLOTLIB VISUALIZATIONS (ALTERNATIVE TO PLOTLY)
# =============================================================================

import matplotlib.pyplot as plt
import pandas as pd
import optuna

# Load the study from database
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STORAGE_FILE}"
)

# Create visualization directory
viz_dir = HPO_DIR / "visualizations"
viz_dir.mkdir(exist_ok=True)

print("\n" + "=" * 80)
print("CREATING MATPLOTLIB VISUALIZATIONS")
print("=" * 80)

# Get trial data
trials_df = study.trials_dataframe()

# 1. Optimization History
plt.figure(figsize=(12, 6))
plt.plot(trials_df['number'], -trials_df['value'], 'o-', linewidth=2, markersize=6)
plt.axhline(y=-study.best_value, color='r', linestyle='--', label=f'Best: {-study.best_value:.4f}')
plt.xlabel('Trial Number', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('Optimization History', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(viz_dir / "optimization_history_mpl.png", dpi=150)
plt.show()
print("✓ Optimization history (matplotlib)")

# 2. Parameter values across trials
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Hyperparameter Values Across Trials', fontsize=14, fontweight='bold')

params_to_plot = ['dropout', 'l2_reg', 'learning_rate', 'batch_size', 'channels_multiplier']
for idx, param in enumerate(params_to_plot):
    ax = axes[idx // 3, idx % 3]
    col_name = f'params_{param}'
    if col_name in trials_df.columns:
        ax.scatter(trials_df['number'], trials_df[col_name], alpha=0.6, s=50)
        ax.set_xlabel('Trial')
        ax.set_ylabel(param)
        ax.grid(True, alpha=0.3)
        
        # Highlight best trial
        best_val = trials_df[trials_df['number'] == study.best_trial.number][col_name].values[0]
        ax.axhline(y=best_val, color='r', linestyle='--', linewidth=2, label=f'Best: {best_val}')
        ax.legend()

# Augmentation distribution
ax = axes[1, 2]
aug_counts = trials_df['params_augmentation'].value_counts()
ax.bar(aug_counts.index, aug_counts.values, alpha=0.7)
ax.set_xlabel('Augmentation')
ax.set_ylabel('Count')
ax.set_title('Augmentation Distribution')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(viz_dir / "parameters_distribution_mpl.png", dpi=150)
plt.show()
print("✓ Parameters distribution (matplotlib)")

# 3. Top 10 trials comparison
top_trials = trials_df.nlargest(10, 'value')[['number', 'value', 'params_dropout', 'params_l2_reg', 'params_learning_rate', 'params_batch_size']]
top_trials['accuracy'] = -top_trials['value']

plt.figure(figsize=(12, 6))
plt.barh(top_trials['number'].astype(str), top_trials['accuracy'], alpha=0.7)
plt.xlabel('Validation Accuracy', fontsize=12)
plt.ylabel('Trial Number', fontsize=12)
plt.title('Top 10 Trials by Accuracy', fontsize=14, fontweight='bold')
plt.axvline(x=-study.best_value, color='r', linestyle='--', linewidth=2, label=f'Best: {-study.best_value:.4f}')
plt.legend()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(viz_dir / "top_trials_mpl.png", dpi=150)
plt.show()
print("✓ Top trials comparison (matplotlib)")

print(f"\n✓ All matplotlib visualizations saved to: {viz_dir}")
print("=" * 80)


ModuleNotFoundError: No module named 'optuna'

In [9]:
# =============================================================================
# OPTUNA VISUALIZATIONS
# =============================================================================
best_params_file = HPO_DIR / "best_parameters.json"
# Create visualization directory
viz_dir = HPO_DIR / "visualizations"
viz_dir.mkdir(exist_ok=True)

print("Generating Optuna visualizations...")

# 1. Optimization history
try:
    fig = optuna.visualization.plot_optimization_history(study)
    fig.write_image(str(viz_dir / "optimization_history.png"))
    fig.show()
    print("✓ Optimization history")
except Exception as e:
    print(f"⚠ Could not create optimization history: {e}")

# 2. Parameter importances
try:
    fig = optuna.visualization.plot_param_importances(study)
    fig.write_image(str(viz_dir / "param_importances.png"))
    fig.show()
    print("✓ Parameter importances")
except Exception as e:
    print(f"⚠ Could not create param importances: {e}")

# 3. Parallel coordinate plot
try:
    fig = optuna.visualization.plot_parallel_coordinate(study)
    fig.write_image(str(viz_dir / "parallel_coordinate.png"))
    fig.show()
    print("✓ Parallel coordinate plot")
except Exception as e:
    print(f"⚠ Could not create parallel coordinate: {e}")

# 4. Slice plot
try:
    fig = optuna.visualization.plot_slice(study)
    fig.write_image(str(viz_dir / "slice_plot.png"))
    fig.show()
    print("✓ Slice plot")
except Exception as e:
    print(f"⚠ Could not create slice plot: {e}")

print(f"\nVisualizations saved to: {viz_dir}")
print("\n" + "=" * 80)
print("✓ HPO COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"\nNext steps:")
print(f"1. Check best parameters in: {best_params_file}")
print(f"2. Review visualizations in: {viz_dir}")
print(f"3. Use best parameters to train final model")
print(f"4. Evaluate on test set")
print("=" * 80)


Generating Optuna visualizations...
⚠ Could not create optimization history: name 'optuna' is not defined
⚠ Could not create param importances: name 'optuna' is not defined
⚠ Could not create parallel coordinate: name 'optuna' is not defined
⚠ Could not create slice plot: name 'optuna' is not defined

Visualizations saved to: ..\..\runs\hpo\visualizations

✓ HPO COMPLETED SUCCESSFULLY!

Next steps:
1. Check best parameters in: ..\..\runs\hpo\best_parameters.json
2. Review visualizations in: ..\..\runs\hpo\visualizations
3. Use best parameters to train final model
4. Evaluate on test set
